# Gen 5 Sub-Agent Benchmark Lab

**Project goal:** optimize sub-agent utilization and build a way for sub-agents to work *asynchronously*
with an orchestrator. Along the way, benchmark every Claude Gen 5 model at every effort level to discover
where each one earns its cost.

This notebook has four parts:

| Part | What it does | Output |
|---|---|---|
| 1. Setup | Install the SDK, load the key, define the Gen 5 model registry and pricing | `MODELS` |
| 2. Model × effort sweep | Run a frozen task suite across models and efforts, score it, price it | `sweep_df` + Pareto chart |
| 3. Async sub-agent orchestrator | Orchestrator with `spawn_subagent` / `check_subagents` / `wait_for_subagents` tools. Compare **solo**, **blocking**, and **async** delegation | `orch_df` |
| 4. Findings | Auto-generated "best config per task category" table and a findings log to fill | `recommend_df` |

**Rules the harness follows** (they come from the current Claude API docs and matter for Gen 5):

- Thinking is adaptive and always on. `thinking` is omitted and depth is controlled with `output_config.effort`.
- `budget_tokens`, `temperature`, `top_p`, `top_k` are never sent (400 on Gen 5).
- Forced `tool_choice` is never used. Tools are `auto` + `strict: true`.
- Histories are append-only and assistant `content` is echoed back unchanged (preserved thinking).
- `stop_reason == "refusal"` is checked before reading content. Server-side fallbacks are on by default (see `USE_FALLBACKS`).
- Every run records `usage`, latency, served model and cost so results are comparable.

> Every cell that calls the API spends real money. Each sweep prints a cost estimate first and runs only when you set `GO = True`.

## Part 1 · Setup

In [ ]:
# @title 1.1 Install
%pip install -q "anthropic>=1.1.0" pandas matplotlib pydantic nest_asyncio
import anthropic, sys
print("anthropic", anthropic.__version__, "| python", sys.version.split()[0])

In [ ]:
# @title 1.2 API key (Colab secret `ANTHROPIC_API_KEY`, else prompt)
import os, getpass

def _load_key() -> str:
    if os.environ.get("ANTHROPIC_API_KEY"):
        return os.environ["ANTHROPIC_API_KEY"]
    try:
        from google.colab import userdata  # type: ignore
        key = userdata.get("ANTHROPIC_API_KEY")
        if key:
            return key
    except Exception:
        pass
    return getpass.getpass("ANTHROPIC_API_KEY: ")

os.environ["ANTHROPIC_API_KEY"] = _load_key()
print("key loaded:", os.environ["ANTHROPIC_API_KEY"][:10] + "…")

In [ ]:
# @title 1.3 Imports, clients, run directory
import asyncio, json, re, time, uuid, math, textwrap, traceback
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Optional

import nest_asyncio; nest_asyncio.apply()   # lets `await` and asyncio.run coexist in Colab
import pandas as pd
from pydantic import BaseModel, Field
from anthropic import AsyncAnthropic, Anthropic

aclient = AsyncAnthropic(max_retries=5, timeout=900.0)   # long timeout: Fable turns can run minutes
client  = Anthropic(max_retries=5, timeout=900.0)

RUN_ID  = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
OUT_DIR = Path("results") / RUN_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("results ->", OUT_DIR.resolve())

# Optional: persist results to Google Drive
MOUNT_DRIVE = False  # @param {type:"boolean"}
if MOUNT_DRIVE:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/ClaudeAgents/results") / RUN_ID
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    print("results ->", OUT_DIR)

### 1.4 Gen 5 model registry

Prices are first-party API rates in USD per million tokens. Cache writes are billed at 1.25× input
(5-minute TTL). Haiku 5.5 has a second rate card above 100K-token prompts; the suite stays well under that.

| Model | ID | In | Out | Cache read | Effort default | Notes |
|---|---|---|---|---|---|---|
| Fable 5.1 | `claude-fable-5-1` | 10 | 50 | 0.25 | high | thinking always on, no forced tool_choice |
| Fable 5 | `claude-fable-5` | 10 | 50 | 1.00 | high | predecessor, off by default |
| Opus 5.5 | `claude-opus-5-5` | 4 | 20 | 0.20 | **medium** | thinking can't be disabled |
| Opus 5 | `claude-opus-5` | 5 | 25 | 0.50 | high | off by default |
| Sonnet 5.5 | `claude-sonnet-5-5` | 2 | 10 | 0.20 | high | `between_tools` thinking available ≤ high |
| Sonnet 5 | `claude-sonnet-5` | 2 | 10 | 0.20 | high | off by default |
| Haiku 5.5 | `claude-haiku-5-5` | 0.10 | 0.50 | 0.01 | **medium** | no server-side refusal fallback |

Mythos 5.1 is Project Glasswing only and is excluded.

In [ ]:
# @title 1.4 Model registry + pricing
EFFORTS = ["low", "medium", "high", "xhigh", "max"]

@dataclass(frozen=True)
class ModelSpec:
    key: str
    id: str
    tier: str
    in_price: float          # $/MTok
    out_price: float         # $/MTok
    cache_read_price: float  # $/MTok
    default_effort: str
    server_fallback: bool    # supports fallbacks="default"
    enabled: bool = True

MODELS: dict[str, ModelSpec] = {m.key: m for m in [
    ModelSpec("fable-5.1",  "claude-fable-5-1",  "fable",  10.0, 50.0, 0.25, "high",   True),
    ModelSpec("fable-5",    "claude-fable-5",    "fable",  10.0, 50.0, 1.00, "high",   True,  enabled=False),
    ModelSpec("opus-5.5",   "claude-opus-5-5",   "opus",    4.0, 20.0, 0.20, "medium", True),
    ModelSpec("opus-5",     "claude-opus-5",     "opus",    5.0, 25.0, 0.50, "high",   True,  enabled=False),
    ModelSpec("sonnet-5.5", "claude-sonnet-5-5", "sonnet",  2.0, 10.0, 0.20, "high",   True),
    ModelSpec("sonnet-5",   "claude-sonnet-5",   "sonnet",  2.0, 10.0, 0.20, "high",   False, enabled=False),
    ModelSpec("haiku-5.5",  "claude-haiku-5-5",  "haiku",   0.10, 0.50, 0.01, "medium", False),
]}

def cost_usd(spec: ModelSpec, usage) -> float:
    u = usage if isinstance(usage, dict) else usage.model_dump()
    inp   = u.get("input_tokens", 0) or 0
    out   = u.get("output_tokens", 0) or 0
    cw    = u.get("cache_creation_input_tokens", 0) or 0
    cr    = u.get("cache_read_input_tokens", 0) or 0
    return (inp * spec.in_price + cw * spec.in_price * 1.25
            + cr * spec.cache_read_price + out * spec.out_price) / 1e6

ENABLED = [k for k, m in MODELS.items() if m.enabled]
print("enabled models:", ENABLED)

# Routing policy: category -> {model, effort, evidence}. Empty until Part 5 builds it from measured results.
POLICY: dict = {"version": 1, "default": {"model": "sonnet-5.5", "effort": "medium"}, "categories": {}}

def route(category: str) -> tuple[str, str]:
    # (model_key, effort) for a task category, from the measured policy; falls back to the default
    entry = POLICY["categories"].get(category) or POLICY["default"]
    return entry["model"], entry["effort"]

### 1.5 The call wrapper

One function makes every request in this notebook so results are comparable. It:

- streams (`.stream()` + `get_final_message()`) so long turns never hit HTTP timeouts,
- sets `output_config.effort` and never sends `thinking`, sampling params, or `budget_tokens`,
- opts into `fallbacks="default"` on models that support it (recorded: `served_model` tells you if a fallback ran),
- returns a `CallResult` with usage, latency, cost, stop reason and the raw content blocks.

In [ ]:
# @title 1.5 Call wrapper
USE_FALLBACKS   = True   # @param {type:"boolean"}
SHOW_THINKING   = False  # @param {type:"boolean"}  (display="summarized" costs nothing extra but adds bytes)
MAX_CONCURRENCY = 4      # @param {type:"integer"}

SEM = asyncio.Semaphore(MAX_CONCURRENCY)

@dataclass
class CallResult:
    model_key: str
    effort: str
    text: str
    content: list                # raw blocks, echo these back verbatim in multi-turn loops
    stop_reason: str
    served_model: str
    usage: dict
    latency_s: float
    cost: float
    error: Optional[str] = None
    refusal_category: Optional[str] = None

def _text_of(content) -> str:
    return "".join(b.text for b in content if getattr(b, "type", "") == "text")

def build_params(model_key: str, effort: str, messages: list, *, system=None, tools=None,
                 max_tokens: int = 16000, **extra) -> dict:
    spec = MODELS[model_key]
    p: dict[str, Any] = dict(model=spec.id, max_tokens=max_tokens, messages=messages,
                             output_config={"effort": effort}, **extra)
    if system:
        p["system"] = system
    if tools:
        p["tools"] = tools
    if SHOW_THINKING:
        p["thinking"] = {"type": "adaptive", "display": "summarized"}
    betas = []
    if USE_FALLBACKS and spec.server_fallback:
        betas.append("server-side-fallback-2026-07-01")
        p["fallbacks"] = "default"
    if betas:
        p["betas"] = betas
    return p

async def call(model_key: str, effort: str, messages: list, **kw) -> CallResult:
    spec = MODELS[model_key]
    params = build_params(model_key, effort, messages, **kw)
    t0 = time.perf_counter()
    async with SEM:
        try:
            async with aclient.beta.messages.stream(**params) as stream:
                msg = await stream.get_final_message()
        except Exception as e:  # typed SDK errors are all subclasses of anthropic.APIError
            return CallResult(model_key, effort, "", [], "error", spec.id, {},
                              time.perf_counter() - t0, 0.0, error=f"{type(e).__name__}: {e}")
    usage = msg.usage.model_dump()
    refusal = None
    if msg.stop_reason == "refusal":
        sd = getattr(msg, "stop_details", None)
        refusal = getattr(sd, "category", None) or "unknown"
    return CallResult(model_key, effort, _text_of(msg.content), list(msg.content), msg.stop_reason,
                      msg.model, usage, time.perf_counter() - t0, cost_usd(spec, usage),
                      refusal_category=refusal)

# smoke test (one cheap call)
_r = await call("haiku-5.5", "low", [{"role": "user", "content": "Reply with the single word OK."}], max_tokens=64)
print(_r.text, "|", _r.served_model, "|", f"{_r.latency_s:.2f}s", "|", f"${_r.cost:.6f}", "|", _r.error)

## Part 2 · Model × effort sweep

### 2.1 Task suite

A frozen set of tasks grouped by **category**, each with a deterministic checker where one exists and an
LLM judge otherwise. Categories map to the sub-agent roles we care about:

| Category | What a sub-agent would be doing | Checker |
|---|---|---|
| `classify` | routing / triage | exact match |
| `extract` | structured extraction from text | field match |
| `reason` | multi-step numeric reasoning | exact numeric |
| `code` | write a function that passes hidden tests | executed tests |
| `tool_pick` | choose the right tool with the right args | arg match |
| `brief` | write a precise brief for another agent | judge (rubric) |
| `summarize` | compress a document faithfully | judge (rubric) |

Add your own tasks to `TASKS`. Keep inputs frozen between runs so sweeps are comparable.

In [ ]:
# @title 2.1 Task suite
@dataclass
class Task:
    id: str
    category: str
    prompt: str
    checker: Callable[[str], float]          # returns score in [0, 1]
    system: Optional[str] = None
    tools: Optional[list] = None
    max_tokens: int = 8000
    needs_judge: bool = False
    rubric: Optional[str] = None
    tier: str = "easy"           # "easy" | "hard"  (hard: built to fail at Haiku low)

def _num_in(text: str) -> Optional[float]:
    m = re.findall(r"-?\d+(?:\.\d+)?", text.replace(",", ""))
    return float(m[-1]) if m else None

def exact_num(expected: float, tol: float = 1e-6):
    return lambda t: 1.0 if (_num_in(t) is not None and abs(_num_in(t) - expected) <= tol) else 0.0

def exact_word(expected: str):
    return lambda t: 1.0 if t.strip().strip(".").lower() == expected.lower() else 0.0

def _norm(v) -> str:
    try:
        return f"{float(v):g}"
    except (TypeError, ValueError):
        return str(v).strip().lower()

def _json_in(text: str):
    m = re.search(r"\{.*\}", text, re.S)
    try:
        return json.loads(m.group(0)) if m else None
    except Exception:
        return None

def json_fields(expected: dict):
    def chk(t):
        got = _json_in(t)
        if not isinstance(got, dict):
            return 0.0
        def same(a, b):
            if isinstance(b, list):
                return isinstance(a, list) and sorted(str(x).strip().lower() for x in a) == sorted(str(x).lower() for x in b)
            return _norm(a) == _norm(b)
        hits = sum(1 for k, v in expected.items() if same(got.get(k, ""), v))
        return hits / len(expected)
    return chk

def code_tests(fn_name: str, tests: list[tuple]):
    def chk(t):
        m = re.search(r"```(?:python)?\n(.*?)```", t, re.S)
        src = m.group(1) if m else t
        ns: dict = {}
        try:
            exec(src, ns)                     # trusted benchmark content only
            fn = ns[fn_name]
            ok = sum(1 for args, want in tests if fn(*args) == want)
            return ok / len(tests)
        except Exception:
            return 0.0
    return chk

def judge_placeholder(_t):            # replaced by the LLM judge in 2.3
    return float("nan")

WEATHER_TOOLS = [{
    "name": "get_weather",
    "description": "Get the current weather for a city. Call this whenever the user asks about weather.",
    "input_schema": {"type": "object", "properties": {"city": {"type": "string"}, "unit": {"type": "string", "enum": ["c", "f"]}},
                     "required": ["city", "unit"], "additionalProperties": False},
    "strict": True,
}, {
    "name": "convert_currency",
    "description": "Convert an amount between currencies. Call this whenever the user asks to convert money.",
    "input_schema": {"type": "object", "properties": {"amount": {"type": "number"}, "from_ccy": {"type": "string"}, "to_ccy": {"type": "string"}},
                     "required": ["amount", "from_ccy", "to_ccy"], "additionalProperties": False},
    "strict": True,
}]

def tool_pick(expected_name: str, expected_args: dict):
    def chk(content):
        for b in content:
            if getattr(b, "type", "") == "tool_use" and b.name == expected_name:
                inp = b.input if isinstance(b.input, dict) else json.loads(b.input)
                hits = sum(1 for k, v in expected_args.items() if _norm(inp.get(k, "")) == _norm(v))
                return hits / len(expected_args)
        return 0.0
    chk.wants_content = True
    return chk

TERSE = "Answer with only the final answer. No explanation."

DOC_A = textwrap.dedent('''
    Quarterly update, Northwind Logistics. Revenue rose 12% to $48.2M driven by the Midwest expansion.
    Operating margin fell from 9.1% to 7.4% because fuel costs grew 31%. Headcount is 1,240, up 6%.
    The board approved a $15M warehouse automation program expected to cut handling cost per parcel by 18%
    by Q3 next year. Customer churn held at 2.1%. The CFO flagged FX exposure in the Canadian contracts.
''').strip()

TASKS: list[Task] = [
    # classify
    Task("cls_1", "classify", "Classify the sentiment of this review as positive, negative, or neutral: "
         "'The battery died after two days and support never replied.'", exact_word("negative"), system=TERSE, max_tokens=64),
    Task("cls_2", "classify", "Classify the sentiment of this review as positive, negative, or neutral: "
         "'Arrived on time. Does what it says.'", exact_word("neutral"), system=TERSE, max_tokens=64),
    Task("cls_3", "classify", "Route this support ticket to one team: billing, technical, or sales. Ticket: "
         "'I was charged twice this month for the same plan.'", exact_word("billing"), system=TERSE, max_tokens=64),
    # extract
    Task("ext_1", "extract", "Extract fields as JSON with keys name, email, plan, demo_requested (true/false): "
         "'Jane Doe (jane@co.com) wants the Enterprise plan and asked for a demo next week.'",
         json_fields({"name": "Jane Doe", "email": "jane@co.com", "plan": "Enterprise", "demo_requested": "true"}),
         system="Reply with a single JSON object and nothing else.", max_tokens=300),
    Task("ext_2", "extract", f"From the document below, extract JSON with keys revenue_musd (number), margin_pct (number), headcount (integer), churn_pct (number).\n\n{DOC_A}",
         json_fields({"revenue_musd": "48.2", "margin_pct": "7.4", "headcount": "1240", "churn_pct": "2.1"}),
         system="Reply with a single JSON object and nothing else.", max_tokens=300),
    # reason
    Task("rsn_1", "reason", "A tank fills at 12 L/min and drains at 5 L/min. It starts with 40 L and has capacity 250 L. "
         "After how many whole minutes is it first at or above 200 L?", exact_num(23), system=TERSE, max_tokens=4000),
    Task("rsn_2", "reason", "A store marks an item up 25% then discounts it 20%, then adds 8% tax. "
         "If the final price is $108.00, what was the original price in dollars?", exact_num(100), system=TERSE, max_tokens=4000),
    Task("rsn_3", "reason", "How many integers between 1 and 1000 inclusive are divisible by 3 or 5 but not by 15?", exact_num(401), system=TERSE, max_tokens=4000),
    # code
    Task("cod_1", "code", "Write a Python function `rle(s: str) -> str` that run-length encodes a string, e.g. 'aaabcc' -> 'a3b1c2'. "
         "Empty string returns ''. Return only a python code block.",
         code_tests("rle", [(("aaabcc",), "a3b1c2"), (("",), ""), (("abc",), "a1b1c1"), (("zzzz",), "z4")]), max_tokens=4000),
    Task("cod_2", "code", "Write a Python function `top_k_frequent(words: list[str], k: int) -> list[str]` returning the k most frequent words, "
         "ties broken alphabetically. Return only a python code block.",
         code_tests("top_k_frequent", [((["a", "b", "a", "c", "b", "a"], 2), ["a", "b"]), ((["x", "y", "y", "x"], 1), ["x"]), (([], 3), [])]), max_tokens=4000),
    Task("cod_3", "code", "Write a Python function `merge_intervals(iv: list[list[int]]) -> list[list[int]]` that merges overlapping closed intervals and returns them sorted. "
         "Return only a python code block.",
         code_tests("merge_intervals", [(([[1, 3], [2, 6], [8, 10]],), [[1, 6], [8, 10]]), (([[1, 4], [4, 5]],), [[1, 5]]), (([],), [])]), max_tokens=4000),
    # tool_pick
    Task("tool_1", "tool_pick", "What's the weather in Lisbon right now, in celsius?", tool_pick("get_weather", {"city": "Lisbon", "unit": "c"}), tools=WEATHER_TOOLS, max_tokens=1000),
    Task("tool_2", "tool_pick", "How much is 250 euros in Japanese yen?", tool_pick("convert_currency", {"amount": "250", "from_ccy": "EUR", "to_ccy": "JPY"}), tools=WEATHER_TOOLS, max_tokens=1000),
    # brief (judge)
    Task("brf_1", "brief", "You are a lead engineer. Write a brief for a sub-agent to audit a Flask app for SQL injection. The sub-agent has read-only repo access and "
         "cannot ask follow-up questions. Keep it under 200 words.", judge_placeholder, needs_judge=True, max_tokens=2000,
         rubric="Score 1 if the brief states the goal, the scope (which files/paths), what evidence to collect, the output format, and what NOT to do, "
                "all without ambiguity and under ~200 words. Deduct for missing items or vagueness."),
    # summarize (judge)
    Task("sum_1", "summarize", f"Summarize this update in three bullet points for a board member.\n\n{DOC_A}", judge_placeholder, needs_judge=True, max_tokens=2000,
         rubric="Score 1 if the summary is exactly three bullets, numerically faithful (12%/48.2M, margin 9.1->7.4, fuel +31%, $15M automation / 18% cut, churn 2.1%, FX risk), "
                "and prioritizes what a board member cares about. Deduct for any wrong number or invented fact."),
]

# ---- hard tier: distractor tools, traps, multi-step reasoning, edge-case code ----
HARD_TOOLS = WEATHER_TOOLS + [{
    "name": "get_forecast",
    "description": "Get the weather forecast for a city for the next N days. Call this for any question about future weather.",
    "input_schema": {"type": "object", "properties": {"city": {"type": "string"}, "days": {"type": "integer"}},
                     "required": ["city", "days"], "additionalProperties": False}, "strict": True,
}, {
    "name": "get_exchange_rate",
    "description": "Get today's exchange rate between two currencies (no amount conversion).",
    "input_schema": {"type": "object", "properties": {"from_ccy": {"type": "string"}, "to_ccy": {"type": "string"}},
                     "required": ["from_ccy", "to_ccy"], "additionalProperties": False}, "strict": True,
}]

DOC_B = textwrap.dedent('''
    Internal memo, Helios Retail, FY planning. Draft figures: revenue $212M, 38 stores, headcount 2.9k.
    Note from finance (supersedes the draft above): audited revenue is $208.4M after returns; store count is 36
    following the two closures in Leeds and Perth. Headcount confirmed at 2,870. Online share of revenue rose
    from 21% to 27%. Regions in operation: UK, Ireland, Australia. The Netherlands launch is deferred to next year.
    Gross margin 41.2% (prior 43.0%); the decline is attributed to markdowns. Inventory days: 64.
''').strip()

TASKS += [
    # classify (hard): sarcasm, mixed sentiment, ambiguous routing
    Task("cls_h1", "classify", "Classify the sentiment of this review as positive, negative, or neutral: "
         "'Great, another update that wiped my settings. Love spending my Sunday on this.'", exact_word("negative"), system=TERSE, max_tokens=64, tier="hard"),
    Task("cls_h2", "classify", "Classify the overall sentiment as positive, negative, or neutral: "
         "'Battery life is genuinely excellent and the camera is fine, but the screen cracked in week one and I would not buy it again.'",
         exact_word("negative"), system=TERSE, max_tokens=64, tier="hard"),
    Task("cls_h3", "classify", "Route this ticket to one team: billing, technical, or sales. Ticket: "
         "'Since Tuesday's release our SSO login loops back to the sign-in page; our renewal is next month and this is making the case for it harder.'",
         exact_word("technical"), system=TERSE, max_tokens=64, tier="hard"),
    # extract (hard): superseded figures, unit normalisation, lists
    Task("ext_h1", "extract", f"From the memo below extract JSON with keys revenue_musd (number, final audited), stores (integer, current), headcount (integer), "
         f"online_share_pct (number, current), regions (list of strings, in operation only), gross_margin_pct (number).\n\n{DOC_B}",
         json_fields({"revenue_musd": "208.4", "stores": "36", "headcount": "2870", "online_share_pct": "27", "regions": ["UK", "Ireland", "Australia"], "gross_margin_pct": "41.2"}),
         system="Reply with a single JSON object and nothing else.", max_tokens=400, tier="hard"),
    Task("ext_h2", "extract", f"From the document below extract JSON with keys margin_change_pts (number: new minus old operating margin, in percentage points), "
         f"fuel_cost_growth_pct (number), automation_budget_musd (number), handling_cost_cut_pct (number), risk_flag (string: the single risk the CFO flagged, 2 words).\n\n{DOC_A}",
         json_fields({"margin_change_pts": "-1.7", "fuel_cost_growth_pct": "31", "automation_budget_musd": "15", "handling_cost_cut_pct": "18", "risk_flag": "FX exposure"}),
         system="Reply with a single JSON object and nothing else.", max_tokens=400, tier="hard"),
    # reason (hard)
    Task("rsn_h1", "reason", "How many trailing zeros does 125! have?", exact_num(31), system=TERSE, max_tokens=6000, tier="hard"),
    Task("rsn_h2", "reason", "What is the smallest positive integer with exactly 12 positive divisors?", exact_num(60), system=TERSE, max_tokens=6000, tier="hard"),
    Task("rsn_h3", "reason", "What is the sum of all three-digit positive integers whose digits are all distinct and all odd?", exact_num(33300), system=TERSE, max_tokens=8000, tier="hard"),
    # code (hard)
    Task("cod_h1", "code", "Write a Python function `evaluate(expr: str) -> float` that evaluates an arithmetic expression with + - * / , parentheses, "
         "unary minus and standard precedence, without using eval/exec. Return only a python code block.",
         code_tests("evaluate", [(("2+3*4",), 14), (("(2+3)*4",), 20), (("-3+5",), 2), (("10/4",), 2.5), (("2*(3+(4-1))*2",), 24), (("-(2+3)*-2",), 10)]),
         max_tokens=8000, tier="hard"),
    Task("cod_h2", "code", "Write a Python function `parse_duration(s: str) -> int` that converts strings like '1h30m15s', '90m', '2d', '45s' to total seconds; "
         "units d/h/m/s may appear in any order at most once each; empty string returns 0; raise ValueError on anything else. Return only a python code block.",
         code_tests("parse_duration", [(("1h30m15s",), 5415), (("90m",), 5400), (("2d",), 172800), (("45s",), 45), (("",), 0), (("15s1h",), 3615)]),
         max_tokens=8000, tier="hard"),
    Task("cod_h3", "code", "Write a Python function `edit_distance(a: str, b: str) -> int` (Levenshtein: insert, delete, substitute each cost 1). Return only a python code block.",
         code_tests("edit_distance", [(("kitten", "sitting"), 3), (("", "abc"), 3), (("flaw", "lawn"), 2), (("intention", "execution"), 5), (("same", "same"), 0)]),
         max_tokens=8000, tier="hard"),
    # tool_pick (hard): distractor tools
    Task("tool_h1", "tool_pick", "I'm flying to Osaka on Thursday for four days. What should I pack, weather-wise?", tool_pick("get_forecast", {"city": "Osaka"}), tools=HARD_TOOLS, max_tokens=1000, tier="hard"),
    Task("tool_h2", "tool_pick", "What's the euro to yen rate today? I'm just checking, nothing to convert.", tool_pick("get_exchange_rate", {"from_ccy": "EUR", "to_ccy": "JPY"}), tools=HARD_TOOLS, max_tokens=1000, tier="hard"),
    Task("tool_h3", "tool_pick", "Is it cold in Reykjavik at the moment? Give it to me in fahrenheit.", tool_pick("get_weather", {"city": "Reykjavik", "unit": "f"}), tools=HARD_TOOLS, max_tokens=1000, tier="hard"),
    # brief (hard, judge)
    Task("brf_h1", "brief", "You are a lead engineer. Write a brief for a sub-agent that will migrate a 40-file Python package from unittest to pytest. "
         "The sub-agent has write access, no network, cannot ask questions, and must not change tested behaviour. Under 250 words.",
         judge_placeholder, needs_judge=True, max_tokens=2500, tier="hard",
         rubric="Score 1 if the brief states: the goal; scope (which files, what to leave alone); the no-behaviour-change constraint with how to prove it "
                "(run the suite before and after, same pass count); the no-network constraint and its consequence (no new dependencies); "
                "how to handle setUp/tearDown, assert methods and test discovery; the deliverable format (summary of files changed, before/after counts, "
                "anything skipped and why); and what to do when blocked. Deduct for missing items, vagueness, or exceeding ~250 words."),
    # summarize (hard, judge): superseded figures trap
    Task("sum_h1", "summarize", f"Summarize this memo in exactly four bullets for the CEO. Use final figures only.\n\n{DOC_B}", judge_placeholder, needs_judge=True,
         max_tokens=2500, tier="hard",
         rubric="Score 1 if: exactly four bullets; uses the audited/final figures ($208.4M, 36 stores, 2,870 headcount) and NOT the superseded draft figures "
                "($212M, 38, 2.9k); mentions online share 21->27%, gross margin 43.0->41.2% due to markdowns, and the Netherlands deferral; nothing invented. "
                "Any superseded figure presented as current scores at most 0.3."),
]
print(len(TASKS), "tasks |", sum(t.tier == "hard" for t in TASKS), "hard |", sorted({t.category for t in TASKS}))

### 2.2 Sweep configuration and cost estimate

`SWEEP_MODELS × SWEEP_EFFORTS × TASKS × TRIALS` calls. The estimate assumes ~400 output tokens for low-effort runs
and scales with effort; real spend is printed as it goes. Set `QUICK = True` for a smoke run first.

In [ ]:
# @title 2.2 Sweep configuration
QUICK = True  # @param {type:"boolean"}
TRIALS = 1    # @param {type:"integer"}
TIERS = "hard"  # @param ["easy", "hard", "all"]

SWEEP_MODELS  = ENABLED if not QUICK else ["haiku-5.5", "sonnet-5.5"]
SWEEP_EFFORTS = EFFORTS if not QUICK else ["low", "high"]
_tiered       = [t for t in TASKS if TIERS == "all" or t.tier == TIERS]
SWEEP_TASKS   = _tiered if not QUICK else [t for t in _tiered if t.category in ("classify", "reason", "code", "tool_pick")][:4]

_OUT_EST = {"low": 400, "medium": 900, "high": 1800, "xhigh": 3500, "max": 6000}  # rough thinking+text per call
def estimate(models, efforts, tasks, trials):
    total = 0.0
    for mk in models:
        s = MODELS[mk]
        for e in efforts:
            for t in tasks:
                total += trials * (300 * s.in_price + _OUT_EST[e] * s.out_price) / 1e6
    return total

n_calls = len(SWEEP_MODELS) * len(SWEEP_EFFORTS) * len(SWEEP_TASKS) * TRIALS
print(f"{n_calls} calls | rough estimate ${estimate(SWEEP_MODELS, SWEEP_EFFORTS, SWEEP_TASKS, TRIALS):.2f}")
print("models:", SWEEP_MODELS, "| efforts:", SWEEP_EFFORTS, "| tasks:", [t.id for t in SWEEP_TASKS])

In [ ]:
# @title 2.3 LLM judge (Opus 5.5, structured output)
JUDGE_MODEL = "claude-opus-5-5"

class Verdict(BaseModel):
    score: float = Field(ge=0, le=1, description="0..1 against the rubric")
    reason: str

async def judge(task: Task, answer: str) -> tuple[float, str]:
    if not answer.strip():
        return 0.0, "empty answer"
    msg = await aclient.messages.parse(
        model=JUDGE_MODEL, max_tokens=2000, output_config={"effort": "medium"},
        system="You are a strict grader. Score the ANSWER against the RUBRIC. Output only the structured verdict.",
        messages=[{"role": "user", "content": f"TASK:\n{task.prompt}\n\nRUBRIC:\n{task.rubric}\n\nANSWER:\n{answer}"}],
        output_format=Verdict,
    )
    v = msg.parsed_output
    return float(v.score), v.reason

In [ ]:
# @title 2.4 Run the sweep
GO = False  # @param {type:"boolean"}

async def run_one(mk: str, effort: str, task: Task, trial: int) -> dict:
    r = await call(mk, effort, [{"role": "user", "content": task.prompt}],
                   system=task.system, tools=task.tools, max_tokens=task.max_tokens)
    row = dict(run_id=RUN_ID, model=mk, effort=effort, task=task.id, category=task.category, tier=task.tier, trial=trial,
               stop_reason=r.stop_reason, served_model=r.served_model, latency_s=round(r.latency_s, 3),
               cost=r.cost, error=r.error, refusal=r.refusal_category, answer=r.text[:4000], **{f"u_{k}": v for k, v in r.usage.items() if isinstance(v, (int, float))})
    if r.error or r.stop_reason == "refusal":
        row["score"], row["judge_reason"] = 0.0, r.error or f"refusal:{r.refusal_category}"
    elif task.needs_judge:
        row["score"], row["judge_reason"] = await judge(task, r.text)
    elif getattr(task.checker, "wants_content", False):
        row["score"], row["judge_reason"] = task.checker(r.content), ""
    else:
        row["score"], row["judge_reason"] = task.checker(r.text), ""
    return row

async def run_sweep(models, efforts, tasks, trials) -> pd.DataFrame:
    # Complete every request at one (model, effort) before the next, in stable order, so cache behaviour is comparable.
    rows, path = [], OUT_DIR / "sweep.jsonl"
    with open(path, "a") as f:
        for mk in models:
            for e in efforts:
                jobs = [run_one(mk, e, t, k) for t in tasks for k in range(trials)]
                for row in await asyncio.gather(*jobs):
                    rows.append(row); f.write(json.dumps(row) + "\n")
                sub = pd.DataFrame(rows); sub = sub[(sub.model == mk) & (sub.effort == e)]
                print(f"{mk:11s} {e:6s} | mean score {sub.score.mean():.2f} | ${sub.cost.sum():.4f} | {sub.latency_s.mean():.1f}s avg")
    return pd.DataFrame(rows)

if GO:
    sweep_df = await run_sweep(SWEEP_MODELS, SWEEP_EFFORTS, SWEEP_TASKS, TRIALS)
    sweep_df.to_csv(OUT_DIR / "sweep.csv", index=False)
    print("\nsaved", OUT_DIR / "sweep.csv", "| total spend", f"${sweep_df.cost.sum():.4f}")
else:
    print("Set GO = True to run.")

In [ ]:
# @title 2.5 Results: score / cost / latency by model × effort
def load_latest_sweep() -> pd.DataFrame:
    files = sorted(Path("results").glob("*/sweep.jsonl"))
    if not files:
        raise FileNotFoundError("no sweep results yet")
    return pd.concat(pd.read_json(f, lines=True) for f in files)

df = sweep_df if "sweep_df" in globals() else load_latest_sweep()
summary = (df.groupby(["model", "effort"])
             .agg(score=("score", "mean"), cost_per_task=("cost", "mean"), p50_latency=("latency_s", "median"),
                  errors=("error", lambda s: s.notna().sum()), refusals=("refusal", lambda s: s.notna().sum()), n=("score", "size"))
             .reset_index())
summary["effort"] = pd.Categorical(summary["effort"], EFFORTS, ordered=True)
summary = summary.sort_values(["model", "effort"])
display(summary.round(4))

by_cat = df.pivot_table(index=["model", "effort"], columns="category", values="score", aggfunc="mean").round(2)
display(by_cat)
if "tier" in df:
    display(df.pivot_table(index=["model", "effort"], columns="tier", values="score", aggfunc="mean").round(2))

In [ ]:
# @title 2.6 Pareto chart: score vs cost per task
import matplotlib.pyplot as plt

PALETTE = {"fable-5.1": "#2a78d6", "fable-5": "#86b6ef", "opus-5.5": "#eb6834", "opus-5": "#f2a383",
           "sonnet-5.5": "#1baf7a", "sonnet-5": "#7fd4b2", "haiku-5.5": "#eda100"}   # fixed hue per model, never cycled
EFFORT_MARK = {"low": 4, "medium": 6, "high": 8, "xhigh": 10, "max": 12}

def pareto(summary: pd.DataFrame, title="Score vs cost per task (marker size = effort)"):
    fig, ax = plt.subplots(figsize=(8, 5))
    for mk, g in summary.groupby("model"):
        g = g.sort_values("cost_per_task")
        ax.plot(g.cost_per_task, g.score, color=PALETTE.get(mk, "#888"), lw=1.5, alpha=0.6)
        for _, r in g.iterrows():
            ax.scatter(r.cost_per_task, r.score, s=EFFORT_MARK[str(r.effort)] ** 2, color=PALETTE.get(mk, "#888"),
                       edgecolor="white", linewidth=1, zorder=3, label=mk if r.effort == g.effort.iloc[0] else None)
            ax.annotate(str(r.effort), (r.cost_per_task, r.score), xytext=(4, 4), textcoords="offset points", fontsize=7, color="#555")
    ax.set_xscale("log"); ax.set_xlabel("cost per task (USD, log)"); ax.set_ylabel("mean score")
    ax.set_title(title, loc="left"); ax.grid(alpha=0.2); ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, title="model"); plt.tight_layout(); plt.show()

pareto(summary)

# Frontier: configs not dominated on (score up, cost down)
pts = summary.sort_values(["cost_per_task", "score"], ascending=[True, False])
best, frontier = -1, []
for _, r in pts.iterrows():
    if r.score > best:
        frontier.append(r); best = r.score
display(pd.DataFrame(frontier)[["model", "effort", "score", "cost_per_task", "p50_latency"]].round(4))

## Part 3 · Asynchronous sub-agent orchestrator

The guidance for Fable 5.1 and Opus 5.5 is explicit: **sub-agents that communicate asynchronously with the
orchestrator outperform spawn-and-block.** The lead keeps working while sub-agents run, isn't bottlenecked on the
slowest one, and long-lived sub-agents keep their context.

The harness below implements that pattern on the Messages API with three client-side tools:

| Tool | Behaviour |
|---|---|
| `spawn_subagent(brief, model, effort)` | Starts an `asyncio.Task` and **returns immediately** with a handle |
| `check_subagents()` | Non-blocking status of every handle |
| `wait_for_subagents(ids?)` | Blocks until the named (or all) handles finish, returns their results |

Completed results are also **pushed to the lead in the next user message** without being asked for, which is
the "deliver the result in a later user message when ready" shape from the docs. The model will still often choose to
wait, which is why the wait tool exists; the savings come from the cases where it carries on.

Three delegation modes are compared on the same task:

- **solo** – no tools, the lead does everything itself
- **blocking** – `spawn_subagent` awaits the result before returning (classic spawn-and-block)
- **async** – non-blocking spawn + push delivery + optional wait

The history is append-only and assistant content blocks are echoed back unchanged, so the loop is compatible with
preserved thinking on every Gen 5 model.

In [ ]:
# @title 3.1 Sub-agent pool
@dataclass
class SubAgent:
    id: str
    brief: str
    model_key: str
    effort: str
    task: asyncio.Task
    started: float
    result: Optional[CallResult] = None
    delivered: bool = False

    @property
    def done(self): return self.task.done()

SUBAGENT_SYSTEM = (
    "You are a sub-agent working for a lead agent. You cannot ask questions. Complete the brief fully, "
    "then report findings in the format the brief asks for. Report only what you verified or found in the "
    "material provided; say explicitly what you could not determine."
)

class SubAgentPool:
    def __init__(self, default_model="haiku-5.5", default_effort="medium", shared_context: str = ""):
        self.agents: dict[str, SubAgent] = {}
        self.default_model, self.default_effort = default_model, default_effort
        self.shared_context = shared_context   # material the lead can hand to every sub-agent (cached prefix)

    def spawn(self, brief: str, model_key: Optional[str] = None, effort: Optional[str] = None) -> SubAgent:
        mk, ef = model_key or self.default_model, effort or self.default_effort
        sid = f"sa_{len(self.agents)+1}"
        system = [{"type": "text", "text": SUBAGENT_SYSTEM, "cache_control": {"type": "ephemeral"}}]
        if self.shared_context:
            system.insert(0, {"type": "text", "text": self.shared_context})
        task = asyncio.create_task(call(mk, ef, [{"role": "user", "content": brief}], system=system, max_tokens=8000))
        sa = SubAgent(sid, brief, mk, ef, task, time.perf_counter())
        self.agents[sid] = sa
        return sa

    def status(self) -> list[dict]:
        return [{"id": a.id, "model": a.model_key, "effort": a.effort, "status": "done" if a.done else "running",
                 "elapsed_s": round(time.perf_counter() - a.started, 1)} for a in self.agents.values()]

    async def wait(self, ids: Optional[list[str]] = None) -> list[dict]:
        targets = [self.agents[i] for i in (ids or list(self.agents)) if i in self.agents]
        if targets:
            await asyncio.gather(*(a.task for a in targets), return_exceptions=True)
        return [self._payload(a, mark=True) for a in targets]

    def pop_finished(self) -> list[dict]:
        out = [self._payload(a, mark=True) for a in self.agents.values() if a.done and not a.delivered]
        return out

    def running(self) -> list[SubAgent]:
        return [a for a in self.agents.values() if not a.done]

    def _payload(self, a: SubAgent, mark=False) -> dict:
        if a.result is None and a.done:
            try: a.result = a.task.result()
            except Exception as e: a.result = CallResult(a.model_key, a.effort, "", [], "error", "", {}, 0, 0, error=str(e))
        if mark: a.delivered = True
        r = a.result
        return {"id": a.id, "status": "done" if a.done else "running",
                "report": (r.text if r and not r.error else f"ERROR: {r.error if r else 'pending'}"),
                "cost": r.cost if r else 0, "elapsed_s": round(r.latency_s, 1) if r else None}

    def totals(self) -> dict:
        rs = [a.result for a in self.agents.values() if a.result]
        return {"subagents": len(self.agents), "sub_cost": sum(r.cost for r in rs),
                "sub_output_tokens": sum(r.usage.get("output_tokens", 0) for r in rs),
                "sub_latency_sum": sum(r.latency_s for r in rs)}

In [ ]:
# @title 3.2 Orchestrator tools + loop
def orchestrator_tools(mode: str, allowed_models: list[str], routing: bool = False) -> list[dict]:
    spawn_desc = ("Start a sub-agent on an independent subtask. "
                  + ("Returns immediately with a handle; the result is delivered to you automatically in a later "
                     "message when it is ready, so keep working on other things meanwhile. " if mode == "async"
                     else "Blocks until the sub-agent finishes and returns its report. ")
                  + "Brief it precisely the first time: goal, the exact material or item it owns, what to report and in what format.")
    if routing:
        cats = sorted(POLICY["categories"]) + ["other"]
        props = {"brief": {"type": "string"},
                 "category": {"type": "string", "enum": cats,
                              "description": "Kind of work the sub-agent will do. The harness picks the model and effort from the measured routing policy."}}
        required = ["brief", "category"]
    else:
        props = {"brief": {"type": "string"},
                 "model": {"type": "string", "enum": allowed_models, "description": "cheaper models for reading/extraction, stronger for judgment"},
                 "effort": {"type": "string", "enum": EFFORTS}}
        required = ["brief", "model", "effort"]
    tools = [{
        "name": "spawn_subagent", "description": spawn_desc,
        "input_schema": {"type": "object", "properties": props, "required": required, "additionalProperties": False},
        "strict": True}]
    if mode == "async":
        tools += [{
            "name": "check_subagents", "description": "Non-blocking: status of every sub-agent you have started. Do not poll this repeatedly; results are delivered to you automatically.",
            "input_schema": {"type": "object", "properties": {}, "additionalProperties": False}, "strict": True,
        }, {
            "name": "wait_for_subagents",
            "description": "Block until the listed sub-agents (or all of them if the list is empty) finish, and return their reports. Use only when you have nothing else useful to do.",
            "input_schema": {"type": "object", "properties": {"ids": {"type": "array", "items": {"type": "string"}}},
                             "required": ["ids"], "additionalProperties": False}, "strict": True}]
    return tools

AUTONOMY = (
    " You are operating autonomously: nobody can answer questions mid-task, so do not ask for permission or confirmation. "
    "Before ending your turn, check your last paragraph. If it is a plan, a question, or a statement of what you will do next, "
    "do that work now with tool calls instead of ending the turn. End your turn only with the finished deliverable."
)
LEAD_SYSTEM_ASYNC = (
    "You are the lead agent. Delegate independent subtasks to sub-agents and keep working while they run. "
    "Spawn all independent sub-agents in the same turn. Intervene if a sub-agent goes off track or is missing context. "
    "Commit to delegation: never redo a sub-agent's work or re-derive its findings. "
    "When every result you need has arrived, write the final deliverable and stop." + AUTONOMY
)
LEAD_SYSTEM_BLOCKING = (
    "You are the lead agent. Delegate independent subtasks to sub-agents. Spawn all independent sub-agents in the same turn. "
    "Commit to delegation: never redo a sub-agent's work. When every result you need has arrived, write the final deliverable and stop." + AUTONOMY
)
LEAD_SYSTEM_SOLO = "You are a careful analyst. Complete the task fully yourself and write the final deliverable."

@dataclass
class OrchResult:
    mode: str; lead_model: str; lead_effort: str; sub_model: str; sub_effort: str
    final_text: str; wall_s: float; lead_cost: float; sub_cost: float; lead_turns: int
    subagents: int; blocked_s: float; stop_reason: str; error: Optional[str] = None
    stalled: bool = False      # tooled mode ended without ever spawning a sub-agent
    nudges: int = 0            # "continue" messages sent after a text-only early stop
    @property
    def total_cost(self): return self.lead_cost + self.sub_cost

async def run_orchestrator(task_prompt: str, *, mode: str, lead_model: str, lead_effort: str,
                           sub_model: str, sub_effort: str, allowed_models: Optional[list[str]] = None,
                           shared_context: str = "", max_turns: int = 12, verbose: bool = True,
                           routing: bool = False) -> OrchResult:
    assert mode in ("solo", "blocking", "async")
    pool = SubAgentPool(sub_model, sub_effort, shared_context)
    allowed = allowed_models or [sub_model]
    tools = None if mode == "solo" else orchestrator_tools(mode, allowed, routing=routing)

    def _resolve(inp: dict) -> tuple[Optional[str], Optional[str]]:
        # Policy routing: the lead names a category, the harness picks (model, effort) from measured evidence.
        if routing:
            return route(inp.get("category", "other"))
        return inp.get("model"), inp.get("effort")
    system = {"solo": LEAD_SYSTEM_SOLO, "blocking": LEAD_SYSTEM_BLOCKING, "async": LEAD_SYSTEM_ASYNC}[mode]
    if mode != "solo" and not routing:
        system += f" Sub-agent default: model {sub_model}, effort {sub_effort}."
    elif routing:
        system += " For each sub-agent, name the category of work it will do; the harness chooses the model and effort."
    messages: list = [{"role": "user", "content": task_prompt + ("\n\n" + shared_context if shared_context and mode == "solo" else "")}]
    t0, lead_cost, blocked, turns, final, stop, err, nudges = time.perf_counter(), 0.0, 0.0, 0, "", "", None, 0
    MAX_NUDGES = 1

    while turns < max_turns:
        turns += 1
        r = await call(lead_model, lead_effort, messages, system=system, tools=tools, max_tokens=16000)
        lead_cost += r.cost
        if r.error or r.stop_reason == "refusal":
            err, stop = r.error or f"refusal:{r.refusal_category}", r.stop_reason; break
        messages.append({"role": "assistant", "content": r.content})    # echo blocks back unchanged
        stop = r.stop_reason
        tool_uses = [b for b in r.content if b.type == "tool_use"]

        if not tool_uses:
            final = r.text
            if mode != "solo" and not pool.agents and nudges < MAX_NUDGES:
                # Early stop: text-only turn before any delegation. A plain "continue" recovers it (documented behaviour).
                nudges += 1
                messages.append({"role": "user", "content": "Continue. Do the work now: call the tools, then deliver the final result. Do not ask questions."})
                if verbose: print(f"  turn {turns}: text-only early stop -> nudged")
                continue
            if mode == "async" and pool.running():
                # Lead ended its turn while sub-agents are still running: wait, deliver, continue.
                tb = time.perf_counter(); reports = await pool.wait(); blocked += time.perf_counter() - tb
                messages.append({"role": "user", "content": "Sub-agent results arrived:\n" + json.dumps(reports, indent=1)
                                 + "\nIncorporate them and produce the final deliverable."})
                continue
            break

        results, spawned = [], {}
        if mode == "blocking":
            # Fair baseline: start every spawn in this turn first, then block the lead until all of them finish.
            for tu in tool_uses:
                if tu.name == "spawn_subagent":
                    inp = tu.input if isinstance(tu.input, dict) else json.loads(tu.input)
                    spawned[tu.id] = pool.spawn(inp["brief"], *_resolve(inp))
            if spawned:
                tb = time.perf_counter(); await pool.wait([s.id for s in spawned.values()]); blocked += time.perf_counter() - tb
        for tu in tool_uses:
            inp = tu.input if isinstance(tu.input, dict) else json.loads(tu.input)
            try:
                if tu.name == "spawn_subagent":
                    if mode == "blocking":
                        out = pool._payload(spawned[tu.id], mark=True)
                    else:
                        sa = pool.spawn(inp["brief"], *_resolve(inp))
                        out = {"id": sa.id, "status": "running", "note": "result will be delivered to you when ready"}
                elif tu.name == "check_subagents":
                    out = pool.status()
                elif tu.name == "wait_for_subagents":
                    tb = time.perf_counter(); out = await pool.wait(inp.get("ids") or None); blocked += time.perf_counter() - tb
                else:
                    out = {"error": f"unknown tool {tu.name}"}
                results.append({"type": "tool_result", "tool_use_id": tu.id, "content": json.dumps(out)})
            except Exception as e:
                results.append({"type": "tool_result", "tool_use_id": tu.id, "content": f"error: {e}", "is_error": True})
        content: list = results                     # all tool_results in ONE user message
        if mode == "async":
            await asyncio.sleep(0)                  # let finished tasks settle
            pushed = pool.pop_finished()
            if pushed:
                content = content + [{"type": "text", "text": "Sub-agent results arrived:\n" + json.dumps(pushed, indent=1)}]
        messages.append({"role": "user", "content": content})
        if verbose:
            print(f"  turn {turns}: {[tu.name for tu in tool_uses]} | running={len(pool.running())} | lead ${lead_cost:.4f}")

    if pool.running():
        await pool.wait()
    tot = pool.totals()
    return OrchResult(mode, lead_model, lead_effort, sub_model, sub_effort, final, time.perf_counter() - t0,
                      lead_cost, tot["sub_cost"], turns, tot["subagents"], blocked, stop, err,
                      stalled=(mode != "solo" and tot["subagents"] == 0), nudges=nudges)

### 3.3 Fan-out benchmark task

A deliberately parallelizable task: four independent documents, each with facts the final deliverable must contain.
Sub-agents get the documents through the shared context (a cached system prefix), so the orchestrator never has to
paste them into briefs. Coverage of the required facts is checked deterministically and quality by the judge.

In [ ]:
# @title 3.3 Fan-out task + checker
DOCS = {
    "vendor_A": "Vendor A: Acme Cloud. Price $0.021 per GB-month, egress $0.08/GB, SLA 99.95%, regions: 12. Supports object lock. Support response: 4h. Breach notice window: 72h.",
    "vendor_B": "Vendor B: Borealis Storage. Price $0.015 per GB-month, egress $0.12/GB, SLA 99.9%, regions: 5. No object lock. Support response: 24h. Breach notice window: 24h.",
    "vendor_C": "Vendor C: Cirrus Data. Price $0.026 per GB-month, egress free up to 1TB then $0.05/GB, SLA 99.99%, regions: 20. Supports object lock. Support response: 1h. Breach notice window: 48h.",
    "vendor_D": "Vendor D: Delta Vault. Price $0.018 per GB-month, egress $0.09/GB, SLA 99.95%, regions: 8. Supports object lock. Support response: 8h. Breach notice window: 72h.",
}
SHARED_CONTEXT = "Reference documents (one per vendor):\n" + "\n".join(f"[{k}] {v}" for k, v in DOCS.items())

FANOUT_PROMPT = (
    "We are choosing a cloud storage vendor for 50 TB stored with 10 TB/month egress (use 1 TB = 1000 GB). "
    "Hard requirements: object lock and SLA >= 99.95%. "
    "For each of the four vendors in the reference documents, compute monthly cost (storage + egress) and check the two hard requirements. "
    "Then deliver: a table of all four vendors with monthly cost, SLA, object lock, breach notice window; the recommended vendor; and one sentence of justification. "
    "Treat each vendor as an independent subtask."
)
# monthly cost = 50,000 GB * price + egress on 10,000 GB  ->  A 1850, B 1950, C 1750 (9,000 GB billed), D 1800
REQUIRED_FACTS = {
    "A_cost": ["1850", "1,850"], "B_cost": ["1950", "1,950"], "C_cost": ["1750", "1,750"], "D_cost": ["1800", "1,800"],
    "recommend_C": ["Cirrus"], "B_disqualified": ["Borealis"],
}
def fanout_coverage(text: str) -> float:
    t = text.replace("$", "")
    return sum(any(a in t for a in alts) for alts in REQUIRED_FACTS.values()) / len(REQUIRED_FACTS)

FANOUT_RUBRIC = ("Score 1 if: all four vendors tabulated with correct monthly cost (A=$1,850, B=$1,950, C=$1,750, D=$1,800 using 1 TB = 1000 GB), "
                 "Borealis is excluded (no object lock, SLA 99.9%), Cirrus is recommended as cheapest compliant, and the justification is correct. "
                 "Deduct heavily for wrong arithmetic or invented facts.")
FANOUT_TASK = Task("fanout", "orchestrate", FANOUT_PROMPT, fanout_coverage, needs_judge=True, rubric=FANOUT_RUBRIC)
print(SHARED_CONTEXT[:200], "…")

In [ ]:
# @title 3.4 Delegation-mode benchmark
GO_ORCH = False  # @param {type:"boolean"}
ORCH_CONFIGS = [
    # (mode, lead_model, lead_effort, sub_model, sub_effort)
    ("solo",     "opus-5.5", "high",   "haiku-5.5",  "medium"),
    ("blocking", "opus-5.5", "high",   "haiku-5.5",  "medium"),
    ("async",    "opus-5.5", "high",   "haiku-5.5",  "medium"),
    ("async",    "opus-5.5", "high",   "sonnet-5.5", "low"),
    ("async",    "fable-5.1", "high",  "haiku-5.5",  "medium"),
]
ORCH_TRIALS = 1  # @param {type:"integer"}

async def run_orch_bench(configs, trials) -> pd.DataFrame:
    rows, path = [], OUT_DIR / "orchestrator.jsonl"
    with open(path, "a") as f:
        for (mode, lm, le, sm, se) in configs:
            for k in range(trials):
                print(f"\n== {mode} | lead {lm}/{le} | sub {sm}/{se} | trial {k}")
                res = await run_orchestrator(FANOUT_PROMPT, mode=mode, lead_model=lm, lead_effort=le, sub_model=sm, sub_effort=se,
                                             allowed_models=[sm], shared_context=SHARED_CONTEXT)
                cov = fanout_coverage(res.final_text)
                jscore, jreason = (await judge(FANOUT_TASK, res.final_text)) if res.final_text else (0.0, res.error or "no output")
                row = dict(run_id=RUN_ID, trial=k, **{k2: v for k2, v in asdict(res).items() if k2 != "final_text"},
                           total_cost=res.total_cost, coverage=cov, judge_score=jscore, judge_reason=jreason, final_text=res.final_text[:4000])
                rows.append(row); f.write(json.dumps(row) + "\n")
                print(f"   wall {res.wall_s:.1f}s | blocked {res.blocked_s:.1f}s | turns {res.lead_turns} | subs {res.subagents} | "
                      f"${res.total_cost:.4f} | coverage {cov:.2f} | judge {jscore:.2f}")
    return pd.DataFrame(rows)

if GO_ORCH:
    orch_df = await run_orch_bench(ORCH_CONFIGS, ORCH_TRIALS)
    orch_df.to_csv(OUT_DIR / "orchestrator.csv", index=False)
    display(orch_df[["mode", "lead_model", "lead_effort", "sub_model", "sub_effort", "wall_s", "blocked_s", "lead_turns",
                     "subagents", "stalled", "nudges", "lead_cost", "sub_cost", "total_cost", "coverage", "judge_score"]].round(4))
    st = orch_df[orch_df.stalled]
    if len(st):
        print(f"\n{len(st)} stalled run(s) (no sub-agent ever spawned). First-turn text of the first one:\n", st.final_text.iloc[0][:800])
else:
    print("Set GO_ORCH = True to run.")

In [ ]:
# @title 3.5 Wall-clock vs cost by delegation mode
def orch_chart(odf: pd.DataFrame):
    if "stalled" in odf: odf = odf[~odf.stalled]      # stalled runs measure instruction-following, not the delegation mode
    g = odf.groupby(["mode", "lead_model", "sub_model"]).agg(wall_s=("wall_s", "mean"), cost=("total_cost", "mean"), score=("judge_score", "mean"), n=("wall_s", "size")).reset_index()
    g["label"] = g["mode"] + "\n" + g["lead_model"] + "→" + g["sub_model"]
    MODE_COLOR = {"solo": "#eda100", "blocking": "#eb6834", "async": "#1baf7a"}
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, col, lab in zip(axes, ["wall_s", "cost"], ["wall-clock seconds", "total cost (USD)"]):
        bars = ax.bar(g.label, g[col], color=[MODE_COLOR[m] for m in g["mode"]], width=0.6, edgecolor="white")
        ax.bar_label(bars, labels=[f"{v:.1f}" if col == "wall_s" else f"${v:.3f}" for v in g[col]], fontsize=8, padding=2)
        ax.set_title(lab, loc="left"); ax.spines[["top", "right"]].set_visible(False); ax.tick_params(axis="x", labelsize=8)
    plt.tight_layout(); plt.show()
    display(g.round(4))

if "orch_df" in globals():
    orch_chart(orch_df)
else:
    files = sorted(Path("results").glob("*/orchestrator.jsonl"))
    print("no orchestrator results in memory;", f"found {len(files)} saved file(s)" if files else "run 3.4 first")
    if files: orch_chart(pd.concat(pd.read_json(f, lines=True) for f in files))

## Part 4 · Findings

### 4.1 Best configuration per task category

For each category: the highest-scoring config, and the **cheapest config within `TOLERANCE` of that score**.
The second column is the one that matters for sub-agent routing, because a sub-agent's job is usually one category.

In [ ]:
# @title 4.1 Recommendation table
TOLERANCE = 0.05  # @param {type:"number"}

def recommend(df: pd.DataFrame, tol: float = TOLERANCE) -> pd.DataFrame:
    g = df.groupby(["category", "model", "effort"]).agg(score=("score", "mean"), cost=("cost", "mean"), latency=("latency_s", "median")).reset_index()
    out = []
    for cat, sub in g.groupby("category"):
        best = sub.sort_values(["score", "cost"], ascending=[False, True]).iloc[0]
        ok = sub[sub.score >= best.score - tol].sort_values(["cost", "latency"]).iloc[0]
        out.append(dict(category=cat, best=f"{best.model}/{best.effort}", best_score=round(best.score, 2), best_cost=round(best.cost, 5),
                        cheapest_within_tol=f"{ok.model}/{ok.effort}", cwt_score=round(ok.score, 2), cwt_cost=round(ok.cost, 5),
                        saving_x=round(best.cost / ok.cost, 1) if ok.cost else None))
    return pd.DataFrame(out)

recommend_df = recommend(df)
recommend_df.to_csv(OUT_DIR / "recommendations.csv", index=False)
display(recommend_df)

### 4.2 Effort curve per model

Flat curve → the lower setting does this workload's work. Steep → higher effort is earning its cost.
Differences of a task or two are noise on single trials; re-run contested cells with `TRIALS = 3`+.

In [ ]:
# @title 4.2 Effort curves
def effort_curves(df: pd.DataFrame):
    g = df.groupby(["model", "effort"]).agg(score=("score", "mean"), cost=("cost", "mean")).reset_index()
    g["effort"] = pd.Categorical(g["effort"], EFFORTS, ordered=True); g = g.sort_values("effort")
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for mk, s in g.groupby("model"):
        axes[0].plot(s.effort.astype(str), s.score, marker="o", ms=5, lw=1.5, color=PALETTE.get(mk, "#888"), label=mk)
        axes[1].plot(s.effort.astype(str), s.cost, marker="o", ms=5, lw=1.5, color=PALETTE.get(mk, "#888"), label=mk)
    axes[0].set_title("mean score by effort", loc="left"); axes[1].set_title("mean cost per task by effort (USD)", loc="left"); axes[1].set_yscale("log")
    for ax in axes: ax.spines[["top", "right"]].set_visible(False); ax.grid(alpha=0.2)
    axes[0].legend(frameon=False); plt.tight_layout(); plt.show()

effort_curves(df)

### 4.3 Findings log

Fill this in after each run and commit `results/<run_id>/` alongside it. Keep claims tied to a run id.

| Run | Question | Evidence | Conclusion |
|---|---|---|---|
| `RUN_ID` | Does Haiku 5.5 at `low` hold for `classify` and `extract`? | `recommend_df` rows | |
| `RUN_ID` | Where does Sonnet 5.5 `medium` beat Opus 5.5 `low` on cost per solved task? | Pareto frontier | |
| `RUN_ID` | Does async delegation cut wall-clock at equal quality and cost? | `orch_df` wall_s / judge_score | |
| `RUN_ID` | Does the lead choose to wait anyway? How often? | `blocked_s` vs `wall_s` | |
| `RUN_ID` | Does Fable 5.1 as lead spawn more and brief better than Opus 5.5? | `subagents`, judge reasons | |

### Next experiments

1. **Re-run failures at higher effort.** Run everything at `low`, re-run only failures at `high`, compare cost per solved task to running everything at `high`.
2. **Sub-agent effort sweep inside the orchestrator**: hold the lead fixed, sweep `sub_effort` across `low..high` for Haiku and Sonnet.
3. **Long-lived sub-agents**: keep a sub-agent conversation open across subtasks (append-only) and measure cache reads versus fresh spawns.
4. **Per-message effort** (beta `mid-conversation-output-config-2026-07-01`): let the lead run at `high` for planning and drop to `medium` for merge turns without a cache reset.
5. **Task budgets** (beta `task-budgets-2026-03-13`) on the lead to see whether a budget reduces turns without hurting coverage.

## Part 5 · Turn findings into a routing policy

A sweep is only useful if future work uses it. This part persists `recommend_df` as a **routing policy**
(`category → model, effort`) with its evidence attached, and makes three things resolve through it:

1. `route(category)` – any direct call.
2. `run_routed(prompt)` – a task with no known category: a Haiku `low` call labels it, then it runs on the routed config, and **escalates** one rung when a checker says it failed (the "re-run failures at higher effort" pattern).
3. The orchestrator with `routing=True` – the lead names a *category* per sub-agent and the harness picks the model and effort.

Evidence rules when merging a new run into the policy:

- a category entry is replaced only when the new run compared **at least as many models and efforts** and used **at least as many trials** as the entry it replaces,
- every entry records `run_id`, `n_tasks`, `trials`, `models`, `efforts`, so thin evidence stays visible,
- entries whose run did not cover every enabled model and every effort (a `QUICK` run) are tagged `thin: true` and should not be trusted beyond a smoke test.

In [ ]:
# @title 5.1 Build or merge the policy from this run's results
POLICY_PATH = Path("routing_policy.json")   # commit this file to the repo (policies/ folder) when you are happy with it
POLICY_URL  = "https://raw.githubusercontent.com/Thazzeus-ML/ClaudeAgents/claude/dazzling-mayer-5r8o68/policies/routing_policy.json"

def load_policy() -> dict:
    global POLICY
    if POLICY_PATH.exists():
        POLICY = json.loads(POLICY_PATH.read_text())
    else:
        try:
            import urllib.request
            POLICY = json.loads(urllib.request.urlopen(POLICY_URL, timeout=10).read())
            POLICY_PATH.write_text(json.dumps(POLICY, indent=2))
        except Exception:
            pass   # keep the in-memory default
    return POLICY

def _evidence(df: pd.DataFrame, cat: str) -> dict:
    sub = df[df.category == cat]
    return dict(run_id=str(sub.run_id.iloc[0]), n_tasks=int(sub.task.nunique()), trials=int(sub.trial.max()) + 1,
                models=sorted(sub.model.unique()), efforts=sorted(sub.effort.unique(), key=EFFORTS.index),
                tiers=sorted(sub.tier.unique()) if "tier" in sub else ["easy"],
                thin=bool(sub.model.nunique() < len(ENABLED) or sub.effort.nunique() < len(EFFORTS)))

def _stronger(new: dict, old: dict) -> bool:
    return (len(new["models"]) >= len(old["models"]) and len(new["efforts"]) >= len(old["efforts"])
            and new["trials"] >= old["trials"] and new["n_tasks"] >= old["n_tasks"]
            and set(new.get("tiers", ["easy"])) >= set(old.get("tiers", ["easy"])))

def merge_policy(rec: pd.DataFrame, df: pd.DataFrame, policy: dict) -> dict:
    cats = policy.setdefault("categories", {})
    for _, r in rec.iterrows():
        mk, ef = r.cheapest_within_tol.split("/")
        ev = _evidence(df, r.category)
        entry = dict(model=mk, effort=ef, score=float(r.cwt_score), cost=float(r.cwt_cost),
                     best=r.best, best_score=float(r.best_score), **ev)
        old = cats.get(r.category)
        if old is None or _stronger(ev, old):
            cats[r.category] = entry; verdict = "set" if old is None else "replaced"
        else:
            verdict = "kept (existing evidence stronger)"
        print(f"{r.category:12s} -> {mk}/{ef:7s} score {entry['score']:.2f} ${entry['cost']:.5f} | {verdict}{' | THIN' if ev['thin'] else ''}")
    policy["updated"] = RUN_ID
    return policy

load_policy()
POLICY = merge_policy(recommend_df, df, POLICY)
POLICY_PATH.write_text(json.dumps(POLICY, indent=2))
(OUT_DIR / "routing_policy.json").write_text(json.dumps(POLICY, indent=2))
print("\nsaved", POLICY_PATH.resolve())
pd.DataFrame(POLICY["categories"]).T[["model", "effort", "score", "cost", "n_tasks", "tiers", "trials", "thin", "run_id"]]

In [ ]:
# @title 5.2 Route any task: classify → run on the routed config → escalate on failure
class Route(BaseModel):
    category: str = Field(description="one of the known categories, or 'other'")
    reason: str

async def classify_task(prompt: str) -> str:
    cats = sorted(POLICY["categories"]) + ["other"]
    msg = await aclient.messages.parse(
        model=MODELS["haiku-5.5"].id, max_tokens=300, output_config={"effort": "low"},
        system=f"Label the task with exactly one category from {cats}. Output only the structured label.",
        messages=[{"role": "user", "content": prompt[:4000]}], output_format=Route)
    c = msg.parsed_output.category
    return c if c in cats else "other"

LADDER = ["haiku-5.5", "sonnet-5.5", "opus-5.5", "fable-5.1"]   # escalation order on failure

def next_rung(model_key: str, effort: str) -> Optional[tuple[str, str]]:
    # first raise effort within the model (two steps), then move up a model at medium
    i = EFFORTS.index(effort)
    if i + 2 < len(EFFORTS) and effort in ("low", "medium"):
        return model_key, EFFORTS[i + 2]
    j = LADDER.index(model_key) if model_key in LADDER else -1
    return (LADDER[j + 1], "medium") if j + 1 < len(LADDER) else None

async def run_routed(prompt: str, *, category: Optional[str] = None, checker: Optional[Callable[[str], float]] = None,
                     escalate: bool = True, max_rungs: int = 2, **kw) -> dict:
    cat = category or await classify_task(prompt)
    mk, ef = route(cat)
    attempts, total_cost = [], 0.0
    for rung in range(max_rungs + 1):
        r = await call(mk, ef, [{"role": "user", "content": prompt}], **kw)
        total_cost += r.cost
        score = checker(r.text) if checker and not r.error else (None if r.error else 1.0)
        attempts.append(dict(model=mk, effort=ef, score=score, cost=r.cost, latency_s=round(r.latency_s, 2), error=r.error))
        if not escalate or score is None or score >= 1.0:
            break
        nxt = next_rung(mk, ef)
        if nxt is None: break
        mk, ef = nxt
    return dict(category=cat, answer=r.text, total_cost=total_cost, attempts=attempts)

# demo on a task with a checker, and on one with none
demo = await run_routed(TASKS[5].prompt, checker=TASKS[5].checker, system=TERSE, max_tokens=4000)
print(json.dumps({k: v for k, v in demo.items() if k != "answer"}, indent=1), "\nanswer:", demo["answer"][:200])

In [ ]:
# @title 5.3 Orchestrator with policy routing (lead names a category, harness picks model + effort)
GO_ROUTED = False  # @param {type:"boolean"}

if GO_ROUTED:
    res = await run_orchestrator(FANOUT_PROMPT, mode="async", lead_model="opus-5.5", lead_effort="high",
                                 sub_model="haiku-5.5", sub_effort="medium", shared_context=SHARED_CONTEXT, routing=True)
    cov = fanout_coverage(res.final_text); js, jr = await judge(FANOUT_TASK, res.final_text)
    print(f"wall {res.wall_s:.1f}s | blocked {res.blocked_s:.1f}s | subs {res.subagents} | ${res.total_cost:.4f} | coverage {cov:.2f} | judge {js:.2f}")
    row = dict(run_id=RUN_ID, trial=0, **{k: v for k, v in asdict(res).items() if k != "final_text"}, total_cost=res.total_cost,
               coverage=cov, judge_score=js, judge_reason=jr, final_text=res.final_text[:4000], routing=True)
    with open(OUT_DIR / "orchestrator.jsonl", "a") as f: f.write(json.dumps(row) + "\n")
else:
    print("Set GO_ROUTED = True to run the async orchestrator with policy routing.")

### 5.4 Using the policy outside this notebook

`routing_policy.json` is plain data. Commit it under `policies/` and any harness can read it:

- **Messages API code**: `route(category)` is ten lines; copy it with the JSON.
- **Claude Code sub-agents** (`.claude/agents/*.md`): set each agent's `model` from the policy entry for the kind of work it does.
- **Managed Agents**: pick the worker agent's model from the policy when you build the `subagents` roster.

Re-run the full sweep after a model release or a prompt change and merge again. The evidence rules keep a thin run
from overwriting a thorough one.

In [ ]:
# @title 5.5 Export everything from this run
import shutil
for name in ["sweep.csv", "orchestrator.csv", "recommendations.csv", "routing_policy.json"]:
    p = OUT_DIR / name
    print(("✓ " if p.exists() else "– ") + str(p))
if "summary" in globals():
    summary.to_csv(OUT_DIR / "sweep_summary.csv", index=False)
archive = shutil.make_archive(str(OUT_DIR), "zip", OUT_DIR)
print("zip:", archive)
try:
    from google.colab import files  # type: ignore
    files.download(archive)
except Exception:
    pass